
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# MLflow와 에이전트 개발

## 서론

검색 에이전트(retrieval agent)를 구축하는 것은 일반적인 모델을 학습시키는 것과는 다릅니다. 이는 사용자의 질의, 임베딩 모델, 벡터 데이터베이스, 그리고 거대 언어 모델(LLM) 간의 동적인 상호작용을 조율하는 과정을 포함합니다. 본 강의에서는 MLflow 3.0 이상 버전이 이러한 에이전트를 개발, 디버깅 및 관리(govern)하는 데 필요한 기반 구조를 어떻게 제공하는지 살펴봅니다. 단순히 로그를 기록하는 수준을 넘어, 검색 단계의 심층적인 추적 가능성(traceability)을 확보하고 Databricks Unity Catalog를 활용해 에이전트 산출물을 관리하는 방법까지 자세히 알아봅니다.

## 수업 목표

* **MLflow의 핵심 구성 요소** 와 에이전트 개발에서 각 요소의 구체적인 역할을 파악합니다. 
* 프롬프트와 리트리버(Retriever) 설정의 변경 사항을 추적할 수 있도록 **실험(Experiments)** 을 구성합니다.
* 표준 모델 플레버(Flavors)와 **GenAI 전용 플레버(LangChain, PyFunc)** 의 차이점을 이해합니다.
* **MLflow 트레이싱(Tracing)** 을 활용하여 특정 검색 실패 현상(예: 빈 검색 결과, 높은 지연 시간)을 진단합니다. 
* **Unity Catalog Model Registry**를 사용하여 검색 에이전트를 등록하고 관리합니다.

## A. 에이전트를 위한 MLflow의 기초

MLflow는 머신러닝 라이프사이클의 전 과정을 관리하기 위해 개발된 오픈소스 플랫폼입니다. 에이전트 개발 관점에서 MLflow는 모든 설정, 코드 버전, 실행 트레이스(Trace)를 기록하는 중앙 데이터 시스템의 역할을 합니다.

MLflow의 가치를 이해하기 위해 'MLflow가 없는' 상황을 가정해 보겠습니다. 개발자들은 복잡한 체인을 디버깅할 때 흔히 산재된 print() 문이나 기본적인 로그에 의존하곤 합니다. 하지만 이러한 방식은 특정 쿼리가 왜 실패했는지(벡터 검색 시 타임아웃이 발생한 것인지, 임베딩 모델로의 쿼리가 잘못된 것인지, 아니면 추론 오류인지) 파악해야 할 때 한계에 부딪힙니다. 구조화된 추적 시스템이 없다면, 이러한 중간 과정의 실패를 특정 설정 변경과 연관 지어 분석하는 것은 거의 불가능합니다.

반면 **MLflow** 를 사용하면 검색 파라미터부터 최종 생성에 이르기까지 에이전트 동작의 모든 측면이 체계적으로 기록됩니다. 덕분에 "정확히 어떤 설정이 이 고품질의 답변을 만들어냈는가?"라는 질문에 확신을 가지고 답할 수 있게 됩니다.

### A1. MLflow의 구성 요소

구체적인 워크플로우를 살펴보기에 앞서, 플랫폼을 지탱하는 아키텍처의 핵심 축들을 이해하는 것이 중요합니다. MLflow는 단일 도구가 아니라 첫 코드 작성부터 최종 운영 관리까지 에이전트 라이프사이클의 다양한 단계를 처리하는 통합 구성 요소들의 모음입니다.

* **MLflow Tracking:** 파라미터, 코드 버전, 메트릭, 출력 파일을 로깅하는 API 및 UI입니다. 검색 에이전트의 경우, 시스템 프롬프트와 리트리버(Retriever) 설정을 추적하는 작업이 이에 해당합니다.
* **MLflow Tracing:** 에이전트의 계층적 실행 흐름을 캡처하는 전용 관찰성(Observability) 기능으로, 특정 검색 도구 호출을 디버깅하는 데 필수적입니다. 
* **MLflow Models:** 모델을 구축할 때 사용한 라이브러리에 구애받지 않고, 실시간 서빙 등 다양한 다운스트림 도구에서 활용할 수 있도록 모델을 패키징하는 표준 포맷입니다. 
* **MLflow Model Registry:** 모델의 라이브러리 라이프사이클 관리, 버전 제어, 스테이지 전환(예: 스테이징에서 운영 환경으로)을 협업하여 진행할 수 있는 중앙 저장소입니다.

### A2. 실험(Experiments)과 실행(Runs)

구성 요소를 이해했다면, 개발 사이클의 첫 단계는 반복 작업(Iteration)을 체계적으로 정리하는 것입니다. 검색 에이전트를 테스트할 때 수십 가지의 시스템 프롬프트나 청킹 전략을 시도하게 되는데, 구조가 잡혀있지 않으면 금방 혼란에 빠지게 됩니다.

**실험(Experiment)** 은 '고객 지원 검색 에이전트'와 같은 특정 프로젝트의 최상위 논리적 컨테이너 역할을 합니다. 실험 내부의 개별 **실행(Run)**은 특정 시점에서의 에이전트 상태를 포착합니다. MLflow는 매 실행마다 **추론 엔진의 설정** 을 로깅함으로써 재현성 문제를 해결합니다.

* **System Prompts:** 에이전트의 페르소나를 정의하는 구체적인 지침입니다 (예: "당신은 검색된 콘텍스트만을 바탕으로 답변하는 친절한 어시스턴트입니다").  
* **Model Configuration:** `temperature` 및 `max_tokens`와 같은 파라미터입니다.
* **Retriever Settings:** 검색할 청크의 개수(k)나 벡터 유사도 필터링 임계값 등 핵심적인 파라미터입니다.

개발자는 `mlflow.set_experiment()` 를 사용해 이러한 실행 기록이 저장될 작업 공간의 위치를 정의하고, 검색 로직의 반복 실험들을 체계적으로 관리합니다.

### A3. 모델 플레이버(Flavors) 및 래퍼(Wrappers)

실험을 기록하고 최적의 설정을 찾았다면, 이제 배포를 위해 해당 에이전트를 패키징해야 합니다. 단순히 파이썬 스크립트만 저장해서는 의존성 패키지, 환경, 구체적인 로딩 로직 없이는 운영 환경에서 제대로 작동하기 어렵습니다.

**모델 플레이버(Model Flavor)** 는 사용자가 이러한 의존성을 수동으로 처리할 필요 없이, MLflow가 모델을 저장, 로드, 서빙할 수 있도록 지원하는 통합 기능입니다.

* **네이티브 생성형 AI 플레이버(Native GenAI Flavors):** MLflow는 **LangChain**(mlflow.langchain)이나 **OpenAI** 같은 라이브러리를 고유하게 지원합니다. 이러한 플레이버는 검색 체인과 그 구성 요소의 직렬화(Serialization)를 자동으로 처리합니다.
* **PyFunc 플레이버(PyFunc Flavor):** 프로덕션 수준의 검색 에이전트에서는 네이티브 플레이버가 지원하지 않는 커스텀 로직(예: 특정 리랭킹 단계나 동적 필터 적용)이 필요한 경우가 많습니다. 파이썬 함수(PyFunc) 플레이버를 사용하면 predict() 메서드만 노출되어 있다는 조건 하에 어떤 파이썬 코드든 모델로 감쌀(Wrap) 수 있습니다.

**주의:** 검색 에이전트에 PyFunc를 사용할 때는 커스텀 리트리버 코드와 필요한 모든 설정 파일이 로깅되는 아티팩트에 포함되어 있는지 확인해야 합니다.


## B. 관찰 가능성 및 추적

이제 패키징된 에이전트가 준비되었으니, 우리는 새로운 과제에 직면하게 되었습니다. 바로 **이 에이전트가 왜 그렇게 행동하는지 그 이유를 이해하는 것입니다.** 단순히 정확도만 확인하면 되었던 기존 모델과 달리, 검색 에이전트는 사용자, 벡터 데이터베이스, 그리고 LLM 간의 복잡한 상호작용으로 이루어진 '블랙박스'와 같습니다. 이 때문에 기존의 표준적인 디버깅 방식으로는 효과를 보기 어렵습니다.

### B1. 트레이싱(Tracing)의 필요성

만약 사용자가 "원격 근무 규정이 어떻게 되나요?"라고 물었을 때 에이전트가 "잘 모르겠습니다"라고 답한다면, 단순한 텍스트 로그만으로는 그 원인을 파악할 수 없습니다. 검색 도구가 문서를 찾는 데 실패한 걸까요? 검색 속도가 너무 느려 타임아웃이 발생한 걸까요? 아니면 LLM이 검색된 컨텍스트를 무시한 걸까요? **MLflow Tracing** 은 체인 내 모든 단계의 입력과 출력을 기록함으로써, 이러한 실행 그래프를 높은 신뢰도로 시각화하여 보여줍니다.

### B2. 트레이스와 스팬

<!-- <img src="../Includes/images/04-mlflow-tracing-ui.png" alt="MLflow tracing UI" /> -->
![04-mlflow-tracing-ui](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/04-mlflow-tracing-ui.png)

*그림 1. 이 다이어그램은 MLflow의 추적 UI를 보여줍니다.* 

MLflow 추적은 **Trace**와 **Span**를 사용하여 실행 흐름을 시각화합니다.

* **Trace:** 사용자의 초기 질문부터 최종 답변에 이르기까지, 전체 요청의 라이프사이클을 나타냅니다.
* **Span:** 개별 작업 단위를 나타냅니다. 검색(Retrieval) 에이전트의 경우, 일반적으로 "query_embedding", "retrieval_tool", "context_generation"과 같은 구체적인 Span들을 볼 수 있습니다.

Tracing 기능은 지원되는 라이브러리의 경우 **자동 로깅** (예: mlflow.langchain.autolog())을 통해 활성화하거나, 커스텀 검색 함수의 경우 @mlflow.trace 데코레이터를 사용하는 **수동 지시(Manual Instrumentation)**을 통해 활성화할 수 있습니다.

### B3. 검색 실패 진단

Tracing의 가장 핵심적인 가치는 검색 도구의 구체적인 실패 모드를 디버깅하는 데 있습니다. Tracing을 통해 개발자는 일반 로그에서는 보이지 않는 다음과 같은 문제들을 정확히 짚어낼 수 있습니다.

1. **비어 있거나 관련 없는 검색:** **Retriever Span**의 출력을 검사하면 벡터 데이터베이스에서 반환된 청크를 정확히 확인할 수 있습니다. 질문이 적절했음에도 Span 출력이 비어 있거나 부적절한 텍스트를 포함하고 있다면, 문제는 LLM이 아니라 임베딩 모델이나 청크 분할 전략에 있다는 것을 알 수 있습니다.  
2. **AI Search의 지연 시간:** Span은 **지연 시간**(소요 시간)을 기록합니다. 에이전트의 동작이 느릴 때 Trace 폭포수(waterfall) 차트를 확인하면, LLM 생성에는 500ms밖에 걸리지 않은 반면 vector_search Span에서 4초가 소요되었음을 발견할 수 있습니다. 이를 통해 모델이 아닌 데이터베이스 쿼리 최적화에 개발 노력을 집중할 수 있습니다.
3. **컨텍스트가 제공되었음에도 발생하는 환각(Hallucination):** Trace를 통해 Retriever Span이 올바른 문서를 반환했음에도 LLM Span의 출력이 이를 무시하고 있는 것이 확인된다면, 이는 추론 실패로 진단할 수 있습니다. 이는 제공된 컨텍스트를 엄격히 따르도록 시스템 프롬프트를 수정해야 함을 의미합니다.


## C. Unity Catalog를 통한 거버넌스

작동 및 디버깅이 완료된 에이전트를 확보했다면 이제 마지막 관문인 운영 환경 거버넌스에 직면하게 됩니다. 검증 없이 개발자가 운영 엔드포인트에 코드를 직접 배포하도록 방치해서는 안 되며, 기반 데이터에 대한 무분별한 접근도 허용할 수 없으므로 견고한 레지스트리 시스템 구축은 필수적입니다.

### C1. Unity Catalog Model Registry

엔터프라이즈 환경에 배포되는 에이전트는 엄격한 거버넌스와 감독이 필요합니다. **Unity Catalog (UC)** 는 이러한 자산들을 관리하는 중앙 집중식 레지스트리 역할을 합니다. 기존의 Workspace 모델 레지스트리와 달리, UC는 데이터와 AI 자산 전반의 접근 제어를 통합하는 3단계 네임스페이스(`catalog.schema.model`)를 제공합니다.

* **접근 제어:** 기본 AI Search 테이블과 마찬가지로, 등록된 에이전트에 대한 권한(`SELECT`, `EXECUTE`)을 관리할 수 있습니다.  
* **데이터 계보(Lineage)** UC는 에이전트가 어떤 데이터 테이블(AI 검색 인덱스 기준)을 사용했는지 추적하여, 원본 문서부터 배포된 에이전트에 이르기까지의 엔드투엔드 계보를 제공합니다

### C2. 에이전트 로깅 및 등록

에이전트를 제어하는 워크플로우는 특정 시그니처를 사용해 모델을 로깅한 다음 이를 등록하는 과정으로 이루어집니다.

1. **모델 시그니처 정의:** 에이전트는 일반적으로 문자열 입력이나 대화 기록 목록을 받습니다. 서빙 엔드포인트가 요청을 올바르게 검증할 수 있도록 `mlflow.models.ModelSignature`를 사용하여 이러한 입출력 스키마를 정의해야 합니다.
2. **모델 로깅:** `mlflow.langchain.log_model`(또는 적절한 플레이버)을 사용합니다. UI에서 작동 가능한 테스트 위젯을 생성할 수 있도록 **입력 예시(Input Example)** 를 포함하는 것이 좋습니다
3. **등록:** 실험(Experiment)에 로깅이 완료되면 다음 명령어를 사용하여 모델 버전을 Unity Catalog에 등록합니다.
   `mlflow.register_model("runs:/<run_id>/model", "catalog.schema.retrieval_agent")`

**참고:** **검색 도구** 자체(Unity Catalog 함수로 정의된 경우)도 동일한 카탈로그 구조 내에서 관리되어야 일관된 보안 경계를 유지할 수 있습니다.


<!-- <img src="../Includes/images/04-model-registery.png" alt="Model Registery in UC" /> -->

![04-model-registery](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/04-model-registery.png)

*그림 2. 이 도표는 UC 모델 레지스트리 인터페이스를 보여줍니다.* 


*참고:* [Manage models in Unity Catalog Documentation](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle)


## D. 요약

이 수업에서는 MLflow의 검색 중심 Workflows에 맞게 어떻게 적응했는지 설명했습니다. 우리는 **MLflow**의 핵심 구성 요소와 **실험**이 리트리버와 프롬프트의 특정 구성을 어떻게 포착하는지를 정의했습니다. 우리는 검색 실패(검색 결과가 좋지 않음)와 추론 실패(예: 환각)를 구분하는 데 중요한 도구로서 **MLflow 추적**을 탐구했습니다. 마지막으로, 우리는 **Unity Catalog**가 이 에이전트들의 버전 관리를 위한 거버넌스 레지스트리를 제공하는 역할에 대해 다뤘습니다.

이번 단원에서는 검색 중심의 워크플로우에 맞춰 MLflow를 활용하는 방법을 살펴보았습니다. **MLflow의 핵심 구성 요소**를 정의하고, **실험(Experiments)** 을 통해 리트리버(retriever)와 프롬프트의 구체적인 설정을 기록하는 방법을 알아보았습니다. 또한, 검색 실패(검색 결과 불량)와 추론 실패(환각 현상 등)를 구분하는 데 필수적인 도구로서 **MLflow 트레이싱(Tracing)** 을 살펴보았습니다. 마지막으로, 이러한 에이전트들의 버전을 관리하고 통제된 레지스트리를 제공하는 **유니티 카탈로그(Unity Catalog)** 의 역할에 대해 알아보았습니다.

**핵심 내용:**

1. **검색의 필수 요소, 트레이싱:** 중간 검색 스팬(span)의 출력을 확인하지 않고서는 에이전트가 왜 "모르겠습니다"라고 답변했는지 효과적으로 디버깅할 수 없습니다.  
2. **사용자 정의 로직을 위한 PyFunc:** 복잡한 검색 전략을 구현할 때는 맞춤형 재순위화(re-ranking)나 필터링 로직을 캡슐화하기 위해 pyfunc 래퍼가 필요한 경우가 많습니다.
3. **Unity Catalog를 통한 거버넌스:** 원본 문서로의 이력 추적(lineage)과 안전한 접근 제어를 보장하기 위해 에이전트는 유니티 카탈로그(catalog.schema.model)에 등록되어야 합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>